In [ ]:
# pip install pandas

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 9.7/9.7 MB 67.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------------------------- 12.3/12.3 MB 70.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: C:\Users\Mason Coco\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
# pip install scipy

Defaulting to user installation because normal site-packages is not writeable
  Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl.metadata (60 kB)
Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl (36.5 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: C:\Users\Mason Coco\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [9]:
import pandas as pd
import numpy as np
from scipy import stats

"""Calculates Cliff's Delta effect size for between-groups."""
# X is low germane
# Y is high germane
def calculate_cliffs_delta(x, y):

    # Collects length of each condition
    lx, ly = len(x), len(y)

    # Temp var initialized to zero
    matrix = np.zeros((lx, ly))

    # For each item in the low germane
    for i in range(lx):
        # For each item in the high germane
        for j in range(ly):
            # If low germane item greater
            if x[i] > y[j]:
                # Assign matrix value to 1
                matrix[i, j] = 1
            # Else if low germane item less
            elif x[i] < y[j]:
                # Assign matrix value to -1
                matrix[i, j] = -1
    # Calculate cliff delta here
    return np.sum(matrix) / (lx * ly)

# Load data
files = {
    ('Low', 'Train'): 'Summary_LowGermane_psd_results_train.csv',
    ('Low', 'Test'): 'Summary_LowGermane_psd_results_test.csv',
    ('High', 'Train'): 'Summary_HighGermane_psd_results_train.csv',
    ('High', 'Test'): 'Summary_HighGermane_psd_results_test.csv'
}

data_list = []

# files = (cond, phase)
# For each of the four files
for (cond, phase), file in files.items():
    # Read the CSV
    temp_df = pd.read_csv(file)
    # cond - low or high
    temp_df['Condition'] = cond
    # phase - train or test
    temp_df['Phase'] = phase
    # add to empty data list
    data_list.append(temp_df)

# reset index
df = pd.concat(data_list).reset_index(drop=True)

# I only care about these two cols
metrics = ['Pz_Alpha', 'Fz_Theta']

# For between and within analysis
results_between = []
results_within = []

# For each freq band
for metric in metrics:

    ####### BETWEEN CONDITIONS (Low vs High) #######

    # For each phase
    for phase in ['Train', 'Test']:
        # Get items for low and high separately
        low_vals = df[(df['Condition'] == 'Low') & (df['Phase'] == phase)][metric].values
        high_vals = df[(df['Condition'] == 'High') & (df['Phase'] == phase)][metric].values

        # Perform wilcoxon
        # not mann whitney because same ptps in both conditions
        stat, p = stats.wilcoxon(low_vals, high_vals)

        # Compute delta for directionality
        delta = calculate_cliffs_delta(low_vals, high_vals)

        results_between.append({
            'Metric': metric, 'Phase': phase,
            'W-stat': stat, 'p-value': p, 'Cliffs_Delta': delta
        })

    ####### WITHIN CONDITIONS (Train vs Test) #######

    # For each cond
    for cond in ['Low', 'High']:

        # Ensure PIDs match for Wilcoxon Signed Rank
        train_df = df[(df['Condition'] == cond) & (df['Phase'] == 'Train')].sort_values('PID')
        test_df = df[(df['Condition'] == cond) & (df['Phase'] == 'Test')].sort_values('PID')

        # Perform wilcoxon - same ptps
        stat, p = stats.wilcoxon(train_df[metric], test_df[metric])

        # Cliff's Delta is often used for within-subject as well in this context
        delta = calculate_cliffs_delta(train_df[metric].values, test_df[metric].values)

        results_within.append({
            'Metric': metric, 'Condition': cond,
            'W-stat': stat, 'p-value': p, 'Cliffs_Delta': delta
        })

# Fix format for pd
btwn_df = pd.DataFrame(results_between)
wthn_df = pd.DataFrame(results_within)

# Applying Bonferroni (multiplying p by number of tests per metric)

btwn_df['Bonferroni_p'] = (btwn_df['p-value'] * 2).clip(upper=1.0)
wthn_df['Bonferroni_p'] = (wthn_df['p-value'] * 2).clip(upper=1.0)

print("--- BETWEEN CONDITIONS ANALYSIS ---")
print(btwn_df.to_string(index=False))

print("\n--- WITHIN CONDITIONS ANALYSIS ---")
print(wthn_df.to_string(index=False))

--- BETWEEN CONDITIONS ANALYSIS ---
  Metric Phase  W-stat  p-value  Cliffs_Delta  Bonferroni_p
Pz_Alpha Train   267.0 0.612344      0.034602      1.000000
Pz_Alpha  Test   284.0 0.826350      0.032872      1.000000
Fz_Theta Train   149.0 0.010102      0.176471      0.020205
Fz_Theta  Test   235.0 0.293203      0.124567      0.586406

--- WITHIN CONDITIONS ANALYSIS ---
  Metric Condition  W-stat  p-value  Cliffs_Delta  Bonferroni_p
Pz_Alpha       Low   255.0 0.477588      0.086505      0.955177
Pz_Alpha      High   277.0 0.735656      0.100346      1.000000
Fz_Theta       Low   264.0 0.577162      0.138408      1.000000
Fz_Theta      High   291.0 0.919362      0.153979      1.000000


In [10]:
# Add this after your 'df' is created
summary_stats = df.groupby(['Condition', 'Phase'])[metrics].agg(['mean', 'median']).reset_index()
print("--- DESCRIPTIVE STATISTICS ---")
print(summary_stats)

--- DESCRIPTIVE STATISTICS ---
  Condition  Phase   Pz_Alpha               Fz_Theta            
                         mean     median        mean      median
0      High   Test  79.975976  75.571835  269.932471  255.108057
1      High  Train  87.966796  80.249567  303.334794  268.838195
2       Low   Test  82.646319  76.172986  321.401249  283.010627
3       Low  Train  96.330393  76.547585  365.435304  309.723826
